# Limpieza de datos bancarios

Objetivo: practicar transformaciones basicas para dejar una tabla lista para analisis.

## 1) Librerias

In [56]:
!pip install pandas numpy

In [57]:
# Manejo de datos
import pandas as pd

In [58]:
# Operaciones vectorizadas
import numpy as np

## 2) Cargar datos

In [59]:
# Leer CSV con separador ;
data = pd.read_csv("bank.csv", sep=";")

In [60]:
# Ver primeras filas
data.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,NaN,5,may,261,1,-1,0,NaN,no
1,44,technician,single,secondary,no,29,yes,no,NaN,5,may,151,1,-1,0,NaN,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,NaN,5,may,76,1,-1,0,NaN,no
3,47,blue-collar,married,NaN,no,1506,yes,no,NaN,5,may,92,1,-1,0,NaN,no
4,33,NaN,single,NaN,no,1,no,no,NaN,5,may,198,1,-1,0,NaN,no


## 3) Agregar identificador de fila

In [61]:
# Crear id secuencial
data["row_id"] = np.arange(1, len(data) + 1)

## 4) Convertir columnas categoricas

In [62]:
# Categoria
data["job"] = data["job"].astype("category")

In [63]:
# Categoria
data["marital"] = data["marital"].astype("category")

In [64]:
# Categoria
data["education"] = data["education"].astype("category")

In [65]:
# Categoria
data["contact"] = data["contact"].astype("category")

## 5) Pasar mes de texto a numero

In [66]:
# Mapeo de mes a numero
data["month"] = data["month"].map(
    {"jan": 1,
     "feb": 2,
     "mar": 3,
     "apr": 4,
     "may": 5, 
     "jun": 6,
     "jul": 7,
     "aug": 8,
     "sep": 9,
     "oct": 10,
     "nov": 11,
     "dec": 12})

In [67]:
# Revisar columna
data[["month"]].head()

,month
0,5
1,5
2,5
3,5
4,5


## 6) Convertir yes/no a booleano

In [68]:
bool_mapping = {"yes": True, "no": False}

In [69]:
# Booleano
data["default"] = data["default"].map(bool_mapping)
data["housing"] = data["housing"].map(bool_mapping)
data["loan"] = data["loan"].map(bool_mapping)
data["y"] = data["y"].map(bool_mapping)

## 7) Crear segmento sociodemografico

In [70]:
# Lista de condiciones
cond1 = (data["age"] < 30) & (data["marital"] == "single")
cond2 = (data["age"] >= 30) & (data["age"] < 50) & (data["marital"] == "married")
cond3 = (data["age"] >= 30) & (data["age"] < 50) & (data["marital"] == "single")
cond4 = (data["age"] >= 50) & (data["marital"] == "married")
cond5 = (data["age"] >= 50) & (data["marital"] == "single")

# Asignar etiqueta segun condicion
data["segmento_sociodemo"] = np.select([cond1, cond2, cond3, cond4, cond5], ["Joven soltero", "Adulto casado", "Adulto soltero", "Senior casado", "Senior soltero"], default="Otros")

In [71]:
# Agrupar y resumir
data.groupby("segmento_sociodemo")[ ["balance", "y"] ].agg(["count", "mean"])

balance                   y          
                     count         mean  count      mean
segmento_sociodemo                                      
Adulto casado        18102  1278.088996  18102  0.087891
Adulto soltero        8398  1371.499762   8398  0.125268
Joven soltero         3787  1068.645366   3787  0.209665
Otros                 6593  1100.077506   6593  0.112999
Senior casado         7726  1883.866037   7726  0.134740
Senior soltero         605  1787.338843    605  0.109091

## 8) Crear segmento educacional

In [72]:
cond_a = (data["age"] < 30) & (data["education"] == "tertiary")
cond_b = (data["age"] >= 30) & (data["age"] < 50) & (data["education"] == "tertiary")
cond_c = (data["age"] >= 50) & (data["education"] == "tertiary")
cond_d = (data["age"] < 30) & ((data["education"] == "primary") | (data["education"] == "secondary"))
cond_e = (data["age"] >= 30) & (data["age"] < 50) & ((data["education"] == "primary") | (data["education"] == "secondary"))
cond_f = (data["age"] >= 50) & ((data["education"] == "primary") | (data["education"] == "secondary"))

data["segmento_educacional"] = np.select([cond_a, cond_b, cond_c, cond_d, cond_e, cond_f], ["Joven Universitario", "Adulto Universitario", "Senior Universitario", "Joven No Universitario", "Adulto No Universitario", "Senior No Universitario"], default="Otros")

In [73]:
data.groupby("segmento_educacional")[ ["balance", "y"] ].agg(["count", "mean"])

balance                   y          
                          count         mean  count      mean
segmento_educacional                                         
Adulto No Universitario   19408  1083.810078  19408  0.081925
Adulto Universitario       9327  1646.419213   9327  0.137236
Joven No Universitario     3596   814.239989   3596  0.152670
Joven Universitario        1474  1413.697422   1474  0.223202
Otros                      1857  1526.754443   1857  0.135703
Senior No Universitario    7049  1617.705916   7049  0.127961
Senior Universitario       2500  2379.502000   2500  0.154800

## 9) Duplicados

In [74]:
# Crear una copia con una fila duplicada
data_duplicada = pd.concat([data, data.iloc[[1]]], ignore_index=True)

In [75]:
# Ver filas duplicadas completas
data_duplicada[data_duplicada.duplicated(keep=False)]

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,row_id,segmento_sociodemo,segmento_educacional
1,44,technician,single,secondary,False,29,True,False,NaN,5,5,151,1,-1,0,NaN,False,2,Adulto soltero,Adulto No Universitario
45211,44,technician,single,secondary,False,29,True,False,NaN,5,5,151,1,-1,0,NaN,False,2,Adulto soltero,Adulto No Universitario


In [76]:
# Eliminar duplicados completos
data_duplicada.drop_duplicates()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,row_id,segmento_sociodemo,segmento_educacional
0,58,management,married,tertiary,False,2143,True,False,NaN,5,5,261,1,-1,0,NaN,False,1,Senior casado,Senior Universitario
1,44,technician,single,secondary,False,29,True,False,NaN,5,5,151,1,-1,0,NaN,False,2,Adulto soltero,Adulto No Universitario
2,33,entrepreneur,married,secondary,False,2,True,True,NaN,5,5,76,1,-1,0,NaN,False,3,Adulto casado,Adulto No Universitario
3,47,blue-collar,married,NaN,False,1506,True,False,NaN,5,5,92,1,-1,0,NaN,False,4,Adulto casado,Otros
4,33,NaN,single,NaN,False,1,False,False,NaN,5,5,198,1,-1,0,NaN,False,5,Adulto soltero,Otros
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,False,825,False,False,cellular,17,11,977,3,-1,0,NaN,True,45207,Senior casado,Senior Universitario
45207,71,retired,divorced,primary,False,1729,False,False,cellular,17,11,456,2,-1,0,NaN,True,45208,Otros,Senior No Universitario
45208,72,retired,married,secondary,False,5715,False,False,cellular,17,11,1127,5,184,3,success,True,45209,Senior casado,Senior No Universitario
45209,57,blue-collar,married,secondary,False,668,False,False,telephone,17,11,508,4,-1,0,NaN,False,45210,Senior casado,Senior No Universitario


In [77]:
# Unicos por row_id
data_duplicada.drop_duplicates(subset=["row_id"])

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,row_id,segmento_sociodemo,segmento_educacional
0,58,management,married,tertiary,False,2143,True,False,NaN,5,5,261,1,-1,0,NaN,False,1,Senior casado,Senior Universitario
1,44,technician,single,secondary,False,29,True,False,NaN,5,5,151,1,-1,0,NaN,False,2,Adulto soltero,Adulto No Universitario
2,33,entrepreneur,married,secondary,False,2,True,True,NaN,5,5,76,1,-1,0,NaN,False,3,Adulto casado,Adulto No Universitario
3,47,blue-collar,married,NaN,False,1506,True,False,NaN,5,5,92,1,-1,0,NaN,False,4,Adulto casado,Otros
4,33,NaN,single,NaN,False,1,False,False,NaN,5,5,198,1,-1,0,NaN,False,5,Adulto soltero,Otros
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,False,825,False,False,cellular,17,11,977,3,-1,0,NaN,True,45207,Senior casado,Senior Universitario
45207,71,retired,divorced,primary,False,1729,False,False,cellular,17,11,456,2,-1,0,NaN,True,45208,Otros,Senior No Universitario
45208,72,retired,married,secondary,False,5715,False,False,cellular,17,11,1127,5,184,3,success,True,45209,Senior casado,Senior No Universitario
45209,57,blue-collar,married,secondary,False,668,False,False,telephone,17,11,508,4,-1,0,NaN,False,45210,Senior casado,Senior No Universitario


In [78]:
# Unicos por row_id y balance
data_duplicada.drop_duplicates(subset=["row_id", "balance"])

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,row_id,segmento_sociodemo,segmento_educacional
0,58,management,married,tertiary,False,2143,True,False,NaN,5,5,261,1,-1,0,NaN,False,1,Senior casado,Senior Universitario
1,44,technician,single,secondary,False,29,True,False,NaN,5,5,151,1,-1,0,NaN,False,2,Adulto soltero,Adulto No Universitario
2,33,entrepreneur,married,secondary,False,2,True,True,NaN,5,5,76,1,-1,0,NaN,False,3,Adulto casado,Adulto No Universitario
3,47,blue-collar,married,NaN,False,1506,True,False,NaN,5,5,92,1,-1,0,NaN,False,4,Adulto casado,Otros
4,33,NaN,single,NaN,False,1,False,False,NaN,5,5,198,1,-1,0,NaN,False,5,Adulto soltero,Otros
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45206,51,technician,married,tertiary,False,825,False,False,cellular,17,11,977,3,-1,0,NaN,True,45207,Senior casado,Senior Universitario
45207,71,retired,divorced,primary,False,1729,False,False,cellular,17,11,456,2,-1,0,NaN,True,45208,Otros,Senior No Universitario
45208,72,retired,married,secondary,False,5715,False,False,cellular,17,11,1127,5,184,3,success,True,45209,Senior casado,Senior No Universitario
45209,57,blue-collar,married,secondary,False,668,False,False,telephone,17,11,508,4,-1,0,NaN,False,45210,Senior casado,Senior No Universitario


## 10) Filtrar

In [79]:
# Quedarse con menores de 70
adultos = data[(data["age"] < 40) & (data["age"] >= 30)]

In [80]:
# Ver resultado
adultos

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y,row_id,segmento_sociodemo,segmento_educacional
2,33,entrepreneur,married,secondary,False,2,True,True,NaN,5,5,76,1,-1,0,NaN,False,3,Adulto casado,Adulto No Universitario
4,33,NaN,single,NaN,False,1,False,False,NaN,5,5,198,1,-1,0,NaN,False,5,Adulto soltero,Otros
5,35,management,married,tertiary,False,231,True,False,NaN,5,5,139,1,-1,0,NaN,False,6,Adulto casado,Adulto Universitario
19,33,services,married,secondary,False,0,True,False,NaN,5,5,54,1,-1,0,NaN,False,20,Adulto casado,Adulto No Universitario
22,32,blue-collar,single,primary,False,23,True,True,NaN,5,5,160,1,-1,0,NaN,False,23,Adulto soltero,Adulto No Universitario
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
45198,37,management,married,tertiary,False,1428,False,False,cellular,16,11,333,2,-1,0,NaN,False,45199,Adulto casado,Adulto Universitario
45199,34,blue-collar,single,secondary,False,1475,True,False,cellular,16,11,1166,3,530,12,other,False,45200,Adulto soltero,Adulto No Universitario
45200,38,technician,married,secondary,False,557,True,False,cellular,16,11,1556,4,-1,0,NaN,True,45201,Adulto casado,Adulto No Universitario
45202,34,admin.,single,secondary,False,557,False,False,cellular,17,11,224,1,-1,0,NaN,True,45203,Adulto soltero,Adulto No Universitario


# Guardar

## 11) Ideas extra de limpieza antes de guardar

In [81]:
# Estandarizar textos: quitar espacios y pasar a minusculas
for col in data.select_dtypes(include=["object"]).columns:
    data[col] = data[col].str.strip().str.lower()


/var/folders/1b/ylrjkq_n52n_y9f5l6h05h_m0000gn/T/ipykernel_52967/704236308.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in data.select_dtypes(include=["object"]).columns:


In [82]:
# Marcar categorias unknown como faltantes
data["job"] = data["job"].replace("unknown", np.nan)
data["education"] = data["education"].replace("unknown", np.nan)
data["contact"] = data["contact"].replace("unknown", np.nan)
data["poutcome"] = data["poutcome"].replace("unknown", np.nan)


In [83]:
# Validacion basica de rangos
data.loc[(data["day"] < 1) | (data["day"] > 31), "day"] = np.nan
data.loc[(data["month"] < 1) | (data["month"] > 12), "month"] = np.nan


In [84]:
# Revisar faltantes despues de limpieza extra
data[["job", "education", "contact", "poutcome", "day", "month"]].isna().sum()


job            288
education     1857
contact      13020
poutcome     36959
day              0
month            0
dtype: int64

In [85]:
data.to_csv("bank_limpio.csv", index=False)

In [86]:
adultos.to_csv("bank_adultos.csv", index=False)